In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [ ]:
import os
import sys

sys.path.append("..")
sys.path.append("./ALAE")

import random

from comet_ml import Experiment

import numpy as np
import torch
from tqdm import tqdm

from src.costs.lse import MLPLSECost
from src.models.gmm_based import GMMEOT
from src.models.light_sbm import LightSBM
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.utils.train import compute_loss, update_average

In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

In [ ]:
torch.set_default_device(device)
# dtype = torch.float64
dtype = torch.float32
# torch.torch.set_default_dtype(dtype)

## 2. Config

In [ ]:
from configs.gmm_based.cost import MLPLSECostConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [ ]:
# Data Type
X_DIM = 512
Y_DIM = 512
INPUT_DATA = "WOMAN" # "MAN" # MAN, WOMAN, ADULT, CHILDREN
TARGET_DATA = "MAN" # "WOMAN" # MAN, WOMAN, ADULT, CHILDREN

# Data
Q_X_UNPAIRED_SAMPLES = 48786 # 1024
R_Y_UNPAIRED_SAMPLES = 10762 # 1024
P_XY_PAIRED_SAMPLES = 2000 # 128

# Optimizer
LR_PAIRED = 3e-4
LR_UNPAIRED = 1e-3

# Sampler
PAIRED_BATCH_SIZE = 512
UNPAIRED_BATCH_SIZE = 512

# Train
MAX_STEPS = 10000
INIT_BY_SAMPLES = True

# Potential
N_POTENTIALS = 10

# Cost
M_POTENTIALS = 1
LOG_V_M_HIDDEN_CHANNELS = [M_POTENTIALS]
B_M_HIDDEN_CHANNELS = [M_POTENTIALS * Y_DIM]

SEED = 44

In [ ]:
cost_config = MLPLSECostConfig(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    m_potentials=M_POTENTIALS,
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
    + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    seed=SEED, steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [ ]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [ ]:
from src.utils.datasets import get_latents
from src.samplers.base import TensorSampler
from src.utils.paired import get_paired_sampler

In [ ]:
X_train, X_test = get_latents(INPUT_DATA, dtype=dtype)
Y_train, Y_test = get_latents(TARGET_DATA, dtype=dtype)

In [ ]:
X_sampler = TensorSampler(X_train.to(dtype), device=device)
Y_sampler = TensorSampler(Y_train.to(dtype), device=device)

In [ ]:
from_dir = f"./datasets/FFHQ/pairs/{INPUT_DATA}->{TARGET_DATA}"
X_paired_train_ = torch.load(os.path.join(from_dir, f"X_train.pt"), map_location=device, weights_only=True).to(dtype)
Y_paired_train_ = torch.load(os.path.join(from_dir, f"Y_train.pt"), map_location=device, weights_only=True).to(dtype)

X_paired_test_ = torch.load(os.path.join(from_dir, f"X_test.pt"), map_location=device, weights_only=True).to(dtype)
Y_paired_test_ = torch.load(os.path.join(from_dir, f"Y_test.pt"), map_location=device, weights_only=True).to(dtype)

In [ ]:
X_paired_train = X_paired_train_[:2000]
Y_paired_train = Y_paired_train_[:2000]

X_paired_test = X_paired_test_[2000:4000]
Y_paired_test = Y_paired_test_[2000:4000]

In [ ]:
# from_dir = f"./datasets/FFHQ/"
# paired_data = torch.load(os.path.join(from_dir, f"kp_traj_ffhq_gen.pt"), map_location=device, weights_only=True).to(
#     dtype
# )

# X_paired = paired_data[:, -1, :]
# Y_paired = paired_data[:, 0, :]

# N = X_paired.shape[0]
# perm = torch.randperm(N)

# test_size = int(0.1 * N)
# test_idx = perm[:test_size]
# train_idx = perm[test_size:]

# # X_paired_train = X_paired[train_idx]
# # Y_paired_train = Y_paired[train_idx]
# X_paired_train = X_paired
# Y_paired_train = Y_paired

# # X_paired_test = X_paired[test_idx]
# # Y_paired_test = Y_paired[test_idx]
# X_paired_test = X_paired
# Y_paired_test = Y_paired

# print(f"Total pairs: {N}. Train: {X_paired_train.shape[0]}, Test: {X_paired_test.shape[0]}")

In [ ]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, P_XY_PAIRED_SAMPLES, device
)

In [ ]:
Q_X_UNPAIRED_SAMPLES = min(Q_X_UNPAIRED_SAMPLES, X_train.shape[0])
Q_X_UNPAIRED_SAMPLES

In [ ]:
R_Y_UNPAIRED_SAMPLES = min(R_Y_UNPAIRED_SAMPLES, Y_train.shape[0])
R_Y_UNPAIRED_SAMPLES

In [ ]:
if Q_X_UNPAIRED_SAMPLES > 0:
    source_data = X_sampler.sample(Q_X_UNPAIRED_SAMPLES)
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if R_Y_UNPAIRED_SAMPLES > 0:
    target_data = Y_sampler.sample(R_Y_UNPAIRED_SAMPLES)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [ ]:
# from src.costs.lse import BatchedLSECost

In [ ]:
import torch.nn as nn
import torchvision

In [ ]:
cost = MLPLSECost(**cost_config.model_dump())
# cost = BatchedLSECost(u, log_v_m_net, m_potentials=M_POTENTIALS)

In [ ]:
light_gcot_model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    light_gcot_model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [ ]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

## 5. Optimizers initialization

In [ ]:
unpaired_params_to_update = [light_gcot_model._log_w_n, light_gcot_model._a_n, light_gcot_model._log_A_n]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [ ]:
D_opt_paired = torch.optim.Adam(light_gcot_model.cost.parameters(), **opt_paired_config.model_dump())

In [ ]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_ALAE_"
    + f"FROM_{INPUT_DATA}_"
    + f"TO_{TARGET_DATA}_"
    + f"P_XY_PAIRED_{P_XY_PAIRED_SAMPLES}_"
    + f"Q_X_UNPAIRED_{Q_X_UNPAIRED_SAMPLES}_"
    + f"R_Y_UNPAIRED_{R_Y_UNPAIRED_SAMPLES}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"SEED_{SEED}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=P_XY_PAIRED_SAMPLES,
    Q_X_UNPAIRED_SAMPLES=Q_X_UNPAIRED_SAMPLES,
    R_Y_UNPAIRED_SAMPLES=R_Y_UNPAIRED_SAMPLES,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [ ]:
experiment = Experiment(
    project_name="Light-GCOT-ALE",
    auto_output_logging=False,
    parse_args=False,
)
experiment.set_name(EXP_NAME)
experiment.log_parameters(config)

In [ ]:
for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = light_gcot_model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    experiment.log_metric("Unpaired loss", D_loss_unpaired.item(), step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)
    
    output_paired = light_gcot_model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    experiment.log_metric("Paired loss", D_loss_paired.item(), step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, light_gcot_model, 0.99)
        light_gcot_model = model_copy
    else:
        light_gcot_model = light_gcot_model

    experiment.log_metric(
        "Train paired loss",
        compute_loss(light_gcot_model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train),
        step=step,
    )
    experiment.log_metric(
        "Test paired loss",
        compute_loss(light_gcot_model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test),
        step=step,
    )

    experiment.log_metric("-f^c(x)", -output_unpaired["f_c"].mean().item(), step=step)
    experiment.log_metric("-f(y)", -output_unpaired["f"].mean().item(), step=step)
    experiment.log_metric("lam_min(A_n)", torch.min(output_unpaired["A_n"]).item(), step=step)
    experiment.log_metric("lam_max(A_n)", torch.max(output_unpaired["A_n"]).item(), step=step)

    if step % train_config.plot_every == 0:
        torch.save(light_gcot_model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(light_gcot_model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

experiment.end()

# 7. Light-SBM training

In [ ]:
import torch.nn.functional as F

In [ ]:
eps = 0.1
lr = 1e-3

n_potentials = 10
is_diag = True
S_init = 0.1

max_iter = 20000

In [ ]:
light_sbm = LightSBM(dim=X_DIM, n_potentials=n_potentials, epsilon=eps, S_diagonal_init=S_init, is_diagonal=is_diag)

light_sbm.to(device)
light_sbm_opt = torch.optim.Adam(light_sbm.parameters(), lr=lr)

In [ ]:
def train(model, max_iter, eps, opt, batch_size=512, safe_t=1e-2, device=device):
    
    pbar = tqdm(range(1, max_iter + 1))
    
    for i in pbar:
        
        x_0_samples = X_sampler.sample(batch_size).to(device)      
        x_1_samples = Y_sampler.sample(batch_size).to(device)
        
        t = torch.rand([batch_size, 1], device=device) * (1 - safe_t)
        
        x_t = x_1_samples * t + x_0_samples * (1 - t) + torch.sqrt(eps * t * (1 - t)) * torch.randn_like(x_0_samples)
                
        predicted_drift = model.get_drift(x_t, t.squeeze())
        
        loss_plan = (model.get_log_C(x_0_samples) - model.get_log_potential(x_1_samples)).mean()
        
        target_drift = (x_1_samples - x_t) / (1 - t)
        
        loss = F.mse_loss(target_drift, predicted_drift)
        
        opt.zero_grad()
        
        loss.backward()
        
        opt.step()
        
        pbar.set_description(f'Loss : {loss.item()} Plan Loss: {loss_plan.item()}')

In [ ]:
train(light_sbm, max_iter, eps, light_sbm_opt, batch_size=512, safe_t=1e-2, device=device)

# FSBM from checkpoint

In [ ]:
from omegaconf import OmegaConf
from pathlib import Path

sys.path.append("../FSBM")
from fsbm.dataset import get_dist_boundary
from fsbm.utils import restore_model

In [ ]:
TRNSF = "gen"
FSBM_SEED = 2 # 0, 1, 2

ckpt_dir = f"../FSBM/outputs/runs/ffhq_{TRNSF}/"

if FSBM_SEED == 0:
    subdir = "2025.11.27/162837"
elif FSBM_SEED == 1:
    subdir = "2025.12.02/124004"
elif FSBM_SEED == 2:
    subdir = "2025.12.02/174536"
else:
    raise ValueError(f"Unknown SEED: {FSBM_SEED}!")

cfg = OmegaConf.load(os.path.join(ckpt_dir, f"{subdir}/.hydra/config.yaml"))
ckpt_file_path = os.path.join(ckpt_dir, f"{subdir}/checkpoints/last.ckpt")

In [ ]:
## Load model
fsbm_model, cfg = restore_model(ckpt_file_path, device=device)

fsbm_model.eval();

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from alae_ffhq_inference import decode, load_model

In [ ]:
alae_model = load_model("./ALAE/configs/ffhq.yaml", training_artifacts_dir="./ALAE/training_artifacts/ffhq/").to(
    device
).to(dtype)

In [ ]:
def normalize_tensor(tensor: torch.Tensor) -> torch.Tensor:
    normalized = tensor / 2 + 0.5
    return normalized.clamp_(0, 1)

def to_uint8(normalized_tensor: torch.Tensor) -> torch.Tensor:
    return normalized_tensor.mul(255).add_(0.5).clamp_(0, 255).to(torch.uint8)

In [ ]:
x = X_paired_test[:10]
y = Y_paired_test[:10]

In [ ]:
direction = "bwd"
output = fsbm_model.sample(x, log_steps=20, nfe=1000, direction=direction)
y_pred = output["xs"]

In [ ]:
decoded = []

for traj in y_pred:
    decoded.append(normalize_tensor(decode(alae_model, traj)))

In [ ]:
num_show = min(5, y_pred.shape[0])
T = y_pred.shape[1]

fig, axes = plt.subplots(num_show, T, figsize=(2*T, 2*num_show))

for s in range(num_show):
    for t in range(T):
        axes[s, t].imshow(decoded[s][t].permute(1,2,0).cpu().numpy())
        axes[s, t].axis("off")
        if s == 0:
            axes[s, t].set_title(f"t={t}")

plt.tight_layout()
plt.savefig(f'{INPUT_DATA}->{TARGET_DATA}_FSBM_{direction}.png', bbox_inches='tight')
plt.show()

# 8. Metrics

In [ ]:
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [ ]:
EVAL_MODEL_STEP = 10000

In [ ]:
light_gcot_model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_{EVAL_MODEL_STEP}.pt"), map_location=device, weights_only=True))

In [ ]:
def eval_model(model: torch.nn.Module, model_name: str) -> tuple[float, float, float]:
    loss_fid = FrechetInceptionDistance().to(device)
    loss_ssim = StructuralSimilarityIndexMeasure(data_range=(-1.0, 1.0)).to(device)
    loss_lpip = LearnedPerceptualImagePatchSimilarity(net_type='alex').to(device)
    
    model.to(device)
    model.eval()
    alae_model.eval()

    with torch.no_grad():
        sampling_batch_size = 128
        num_samples = min(len(X_test), len(Y_test))
        print(f"Number of X_test samples: {len(X_test)}")
        print(f"Number of Y_test samples: {len(Y_test)}")
        print(f"Using {num_samples} paired samples")

        num_iters = (num_samples + sampling_batch_size - 1) // sampling_batch_size

        for i in tqdm(range(num_iters)):
            start = i * sampling_batch_size
            end   = min(start + sampling_batch_size, num_samples)
        
            # sub_batch_x = X_paired_test[sampling_batch_size * i : sampling_batch_size * (i + 1)]
            # sub_batch_y = Y_paired_test[sampling_batch_size * i : sampling_batch_size * (i + 1)]
            sub_batch_x = X_test[start:end].to(device)
            sub_batch_y = Y_test[start:end].to(device)

            if "FSBM" in model_name:
                output = model.sample(sub_batch_x, log_steps=20, nfe=1000, direction="fwd")
                y_pred = output["xs"][:, -1, :]
            else:
                y_pred = model(sub_batch_x)
            normalized_pred_images = normalize_tensor(decode(alae_model, y_pred))
            normalized_true_images = normalize_tensor(decode(alae_model, sub_batch_y))

            loss_fid.update(to_uint8(normalized_pred_images), real=False)
            loss_fid.update(to_uint8(normalized_true_images), real=True)

            loss_ssim.update(normalized_pred_images, normalized_true_images)
            loss_lpip.update(normalized_pred_images, normalized_true_images)

            # Explicitly free sub-batches to release GPU memory
            del sub_batch_x, sub_batch_y, y_pred, normalized_pred_images, normalized_true_images
            torch.cuda.empty_cache()
    
    loss_fid_out = loss_fid.compute()
    loss_ssim_out = loss_ssim.compute()
    loss_lpip_out = loss_lpip.compute()
    
    torch.save(loss_fid_out, os.path.join(OUTPUT_PATH, f"FID_{model_name}_{EVAL_MODEL_STEP}.pt"))
    torch.save(loss_ssim_out, os.path.join(OUTPUT_PATH, f"SSIM_{model_name}_{EVAL_MODEL_STEP}.pt"))
    torch.save(loss_lpip_out, os.path.join(OUTPUT_PATH, f"LPIP_{model_name}_{EVAL_MODEL_STEP}.pt"))
    
    return loss_fid_out, loss_ssim_out, loss_lpip_out

In [ ]:
# torch.load(os.path.join(OUTPUT_PATH, f"FID_light-gcot_{EVAL_MODEL_STEP}.pt"))

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
loss_fid, loss_ssim, loss_lpip = eval_model(light_gcot_model, f"light-gcot-{SEED}")
print(f"FID: {loss_fid}")
print(f"SSIM: {loss_ssim}")
print(f"LPIPS: {loss_lpip}")

In [ ]:
# SEEDS 42, 43, 44
light_gcot_fids = np.array([9.45474910736084, 9.355756759643555, 9.207294464111328])
light_gcot_ssims = np.array([0.5317203402519226, 0.531522274017334, 0.5312055945396423])
light_gcod_lpips = np.array([0.5529246926307678, 0.5525802969932556, 0.553896427154541])
print(f"FID: {light_gcot_fids.mean()} +- {light_gcot_fids.std()}")
print(f"SSIM: {light_gcot_ssims.mean()} +- {light_gcot_ssims.std()}")
print(f"LPIPS: {light_gcod_lpips.mean()} +- {light_gcod_lpips.std()}")

In [ ]:
loss_fid, loss_ssim, loss_lpip = eval_model(fsbm_model, f"FSBM_{FSBM_SEED}")
print(f"FID: {loss_fid}")
print(f"SSIM: {loss_ssim}")
print(f"LPIPS: {loss_lpip}")

In [ ]:
# SEEDS 0, 1, 2 
fsbm_fids = np.array([9.47238540649414, 11.056487083435059, 10.184301376342773])
fsbm_ssims = np.array([0.5240597724914551, 0.5229406952857971, 0.5239837765693665])
fsbm_lpips = np.array([0.561979353427887, 0.5628049969673157, 0.5625840425491333])
print(f"FID: {fsbm_fids.mean()} +- {fsbm_fids.std()}")
print(f"SSIM: {fsbm_ssims.mean()} +- {fsbm_ssims.std()}")
print(f"LPIPS: {fsbm_lpips.mean()} +- {fsbm_lpips.std()}")

In [ ]:
# loss_fid_light_sbm, loss_ssim_light_sbm, loss_lpip_light_sbm = eval_model(light_sbm, "light-sbm")
# print(f"FID: {loss_fid_light_sbm}")
# print(f"SSIM: {loss_ssim_light_sbm}")
# print(f"LPIPS: {loss_lpip_light_sbm}")

# 8. Plotting

In [ ]:
# Parameters
num_images = 10        # Number of test images to show
num_gen = 5            # Number of generated versions per image
models = [light_gcot_model, fsbm_model]
model_names = ["Our", "FSBM"]

# Select random test samples
x = X_paired_test[indices]
y = Y_paired_test[indices]


# Decode input and target images
init_img = normalize_tensor(decode(alae_model, x))
true_img = normalize_tensor(decode(alae_model, y))

# Generate predictions for each model
all_model_preds = []  # List to hold predictions from each model

for model, model_name in zip(models, model_names):
    model_preds = []
    for _ in range(num_gen):
        with torch.no_grad():
            print(f"Processing: {model_name}")
            if model_name == "FSBM":
                output = model.sample(x, log_steps=20, nfe=1000, direction="fwd")
                y_pred = output["xs"][:, -1, :]
            else:
                y_pred = model(x)
            
            decoded = normalize_tensor(decode(alae_model, y_pred))
            model_preds.append(decoded)
    model_preds = torch.stack(model_preds, dim=1)  # [num_images, num_gen, C, H, W]
    all_model_preds.append(model_preds)

In [ ]:
# Convert to numpy arrays for plotting
init_img_np = init_img.cpu().permute(0, 2, 3, 1).numpy()      # [num_images, H, W, C]
true_img_np = true_img.cpu().permute(0, 2, 3, 1).numpy()      # [num_images, H, W, C]
all_model_preds_np = [
    preds.cpu().permute(0, 1, 3, 4, 2).numpy()                # [num_images, num_gen, H, W, C]
    for preds in all_model_preds
]

In [ ]:
# Plotting
cols = 2 + num_gen * len(models)
fig, axes = plt.subplots(
    num_images,
    cols,
    figsize=(cols, num_images * 1.5),
    dpi=200
)

for i in range(num_images):
    # Input image
    axes[i, 0].imshow(init_img_np[i])
    axes[i, 0].set_title('Input' if i == 0 else '')
    axes[i, 0].axis('off')

    # Target image
    axes[i, 1].imshow(true_img_np[i])
    axes[i, 1].set_title('Target' if i == 0 else '')
    axes[i, 1].axis('off')

    # Generated images from each model
    col_idx = 2
    for m_idx, model_preds in enumerate(all_model_preds_np):
        for g_idx in range(num_gen):
            axes[i, col_idx].imshow(model_preds[i, g_idx])
            if i == 0:
                axes[i, col_idx].set_title(f'{model_names[m_idx]}')
            axes[i, col_idx].axis('off')
            col_idx += 1

plt.tight_layout(pad=0.5)
plt.savefig(f'{INPUT_DATA}->{TARGET_DATA}_full.png', bbox_inches='tight')
plt.close()

In [ ]:
f'{INPUT_DATA}->{TARGET_DATA}_full.png'

In [ ]:
selected_indices = [0, 1, 2, 3]

In [ ]:
# Plotting
cols = 2 + num_gen * len(models)
fig, axes = plt.subplots(
    len(selected_indices),
    cols,
    figsize=(cols, len(selected_indices) * 1.5),
    dpi=200
)

for i in range(num_images):
    # Input image
    if i in selected_indices:
        axes[i, 0].imshow(init_img_np[i])
        axes[i, 0].set_title('Input' if i == 0 else '')
        axes[i, 0].axis('off')

        # Target image
        axes[i, 1].imshow(true_img_np[i])
        axes[i, 1].set_title('Target' if i == 0 else '')
        axes[i, 1].axis('off')

        # Generated images from each model
        col_idx = 2
        for m_idx, model_preds in enumerate(selected_model_preds_np):
            for g_idx in range(num_gen):
                axes[i, col_idx].imshow(model_preds[i, g_idx])
                if i == 0:
                    axes[i, col_idx].set_title(f'{model_names[m_idx]}')
                axes[i, col_idx].axis('off')
                col_idx += 1

plt.tight_layout(pad=0.5)
plt.savefig(f'{INPUT_DATA}->{TARGET_DATA}_selected.png', bbox_inches='tight')
plt.close()

# Saving images

In [ ]:
from tqdm import trange

from torchvision.utils import make_grid, save_image

In [ ]:
def save_row(x, y, model_preds, model_names, num_gen, out_dir, idx):
    os.makedirs(out_dir, exist_ok=True)

    row = [x, y]  # first two images

    # each entry: [batch, num_gen, C,H,W]
    for m_idx, _ in enumerate(model_names):
        for g in range(num_gen):
            row.append(model_preds[m_idx][g])  #  (C,H,W)

    row = torch.stack(row, dim=0)  # (N,C,H,W)
    grid = make_grid(row, nrow=row.shape[0])

    save_path = os.path.join(out_dir, f"row_{idx:05d}.png")
    save_image(grid, save_path)

In [ ]:
def generate_all_rows_batched(
    X_test: torch.Tensor,
    Y_test: torch.Tensor,
    models: list[nn.Module],
    model_names: list[str],
    alae_model: nn.Module,
    batch_size: int = 64,
    num_gen: int = 1,
    out_dir: str = "generated_rows",
    indices: None | list[int] = None,  # NEW
):
    device = next(models[0].parameters()).device  # same GPU for all models

    if indices is not None:
        indices = sorted(list(indices))
        num_samples = len(indices)
    else:
        num_samples = min(len(X_test), len(Y_test))

    num_batches = (num_samples + batch_size - 1) // batch_size

    idx_global = 0  # global counter for saving row_NNNNN.png files

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, num_samples)

        print(f"Batch {batch_idx+1}/{num_batches} → samples {start}:{end}")

        # Load batch
        if indices is None:
            batch_ids = list(range(start, end))
        else:
            batch_ids = indices[start:end]

        x = X_test[batch_ids].to(device)
        y = Y_test[batch_ids].to(device)

        # Decode X and Y (once per batch)
        x_dec = normalize_tensor(decode(alae_model, x))
        y_dec = normalize_tensor(decode(alae_model, y))

        # Predict with each model
        batch_model_preds = []  # list of shape [num_models] -> [batch, num_gen, C,H,W]

        for model, mname in zip(models, model_names):
            preds_gens = []

            for _ in range(num_gen):
                with torch.no_grad():
                    if mname == "FSBM":
                        out = model.sample(x, log_steps=20, nfe=1000, direction="fwd")
                        y_pred = out["xs"][:, -1, :]
                    else:
                        y_pred = model(x)

                    y_decoded = normalize_tensor(decode(alae_model, y_pred))
                    preds_gens.append(y_decoded)

            preds_gens = torch.stack(preds_gens, dim=1)  # [batch, num_gen, C,H,W]
            batch_model_preds.append(preds_gens)

        # Save each sample row
        for local_i in trange(len(batch_ids)):
            real_idx = batch_ids[local_i]

            x_i = x_dec[local_i]
            y_i = y_dec[local_i]

            preds_i = [m[local_i] for m in batch_model_preds]

            save_row(x_i, y_i, preds_i, model_names=model_names, num_gen=num_gen, out_dir=out_dir, idx=real_idx)

            idx_global += 1

        # Clear memory
        del x, y, x_dec, y_dec, batch_model_preds
        torch.cuda.empty_cache()

In [ ]:
models = [fsbm_model, light_gcot_model]
model_names = ["FSBM", "Our"]

generate_all_rows_batched(
    X_paired_test,
    Y_paired_test,
    models=models,
    model_names=model_names,
    alae_model=alae_model,
    batch_size=64,
    num_gen=1,
    out_dir="rows_output_right_order",
    indices=[454]
)